# Mesin Klip (Colab)

Sekali saja: klik ikon kunci (Secrets) di kiri, tambah `GEMINI_API_KEY`, nyalakan akses notebook.

Lalu tempel tautan di bawah dan tekan **Runtime > Run all**. Klip tersimpan di Google Drive, folder `Mesin Klip`.

In [ ]:
#@title 1. Tautan video
url = "" #@param {type:"string"}
jumlah_klip = 2 #@param {type:"integer"}
niche = "agama, psikologi, leadership" #@param {type:"string"}
assert url.strip(), "Isi dulu tautan videonya."

In [ ]:
#@title 2. Siapkan mesin
!pip -q install -U yt-dlp faster-whisper
!rm -rf /content/mesin-klip && git clone -q --depth 1 https://github.com/khomsun96/mesin-klip /content/mesin-klip
from google.colab import drive, userdata
drive.mount("/content/drive")

In [ ]:
#@title 3. Buat klip
import os, subprocess, shutil, glob, torch
os.environ.update({
    "GEMINI_API_KEY": userdata.get("GEMINI_API_KEY"),
    "URL_VIDEO": url.strip().split("?si=")[0],
    "JUMLAH_KLIP": str(jumlah_klip),
    "NICHE": niche,
    "WHISPER_DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "PYTHONUNBUFFERED": "1",
})
proses = subprocess.Popen(["python", "/content/mesin-klip/klip.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for baris in proses.stdout:
    print(baris, end="")
assert proses.wait() == 0, "Gagal. Kirim tangkapan layar pesan di atas ke Claude."
tujuan = "/content/drive/MyDrive/Mesin Klip"
os.makedirs(tujuan, exist_ok=True)
for f in glob.glob("/content/mesin-klip/hasil/*"):
    shutil.copy(f, tujuan)
    print("Tersimpan:", os.path.basename(f))
print(open("/content/mesin-klip/hasil/caption.md").read())